
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lab - Building Vector Search for Retrieval

Welcome to the lab! Follow the steps below to learn how to build a vector search solution for document retrieval using Databricks.

## Overview

In this lab, you will work with document chunks stored in a parquet file to build a complete vector search solution. You will learn how to **prepare data**, **create vector search indexes**, and **perform various types of searches** using Databricks Vector Search capabilities.

## Learning Objectives

By the end of this lab, you will be able to:
1. **Read** parquet data and save as Delta table with Change Data Feed enabled.
1. **Create** a vector search index using the Databricks UI.
1. **Get** the vector search index for performing searches.
1. **Implement** similarity search with reranking for improved precision.
1. **Execute** hybrid search with filtering to target specific documents.

## Requirements
- A pre-created **vector search endpoint**. This is pre-created for you.
- **Serverless Compute (environment version 5)**. Follow the instructions [here](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) to select the appropriate environment version.
- Required libraries are added to **Dependencies** of Serverless compute configuration.
- Appropriate permissions to create and manage vector search indexes.
- Access to Foundation Model APIs for embedding generation.


**📌 Your Task: In this lab, your task is to replace `<FILL_IN>` sections with appropriate code.**

## Setup

Run the code below to install required libraries and configure your classroom environment. This step ensures all dependencies are available and your workspace is ready for the demo.

In [0]:
%run ../Includes/Classroom-Setup-03

## Task 1: Read Parquet Data and Create Delta Table with CDF

In this section, you will **read document chunks from a parquet file** and save them as a Delta table with Change Data Feed (CDF) enabled. CDF is required for vector search synchronization.

**Steps:**
1. Read the parquet file containing document chunks using pandas.
2. Convert to Spark DataFrame and save as a Delta table.
3. Enable Change Data Feed on the table.
4. Display sample data to verify the table structure.

Complete the code below to perform this task.

In [0]:
docs_chunked_lab_3 = f"{catalog}.{schema}.docs_chunked_lab_3"

In [0]:
# Read parquet file and create Delta table with CDF enabled
import os
import pandas as pd

# Define the parquet file path
parquet_path = f"/Volumes/{catalog}/{schema}/orion_text/docs_chunked.parquet"

# Read Parquet file using pandas
pdf = <FILL_IN>

# Drop the table if it already exists to avoid conflicts
spark.sql(f"DROP TABLE IF EXISTS {docs_chunked_lab_3}")

# Convert pandas DataFrame to Spark DataFrame and write to Unity Catalog
df = spark.createDataFrame(pdf)
df.write.<FILL_IN>

# Enable Change Data Feed for vector search synchronization
spark.sql(f"<FILL_IN>")

print(f"👍 Table '{docs_chunked_lab_3}' created with Change Data Feed enabled.")

In [0]:
%skip
# Read parquet file and create Delta table with CDF enabled
import os
import pandas as pd

# Define the parquet file path
parquet_path = f"/Volumes/{catalog}/{schema}/orion_text/docs_chunked.parquet"

# Read Parquet file using pandas
pdf = pd.read_parquet(parquet_path)

# Drop the table if it already exists to avoid conflicts
spark.sql(f"DROP TABLE IF EXISTS {docs_chunked_lab_3}")

# Convert pandas DataFrame to Spark DataFrame and write to Unity Catalog
df = spark.createDataFrame(pdf)
df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(docs_chunked_lab_3)

# Enable Change Data Feed for vector search synchronization
spark.sql(f"ALTER TABLE {docs_chunked_lab_3} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"👍 Table '{docs_chunked_lab_3}' created with Change Data Feed enabled.")

In [0]:
# Display sample data to understand table structure
display(spark.sql(f"SELECT * FROM {docs_chunked_lab_3} LIMIT 5"))

## Task 2: Create Vector Search Index Using the UI

In this section, you will **create a vector search index using the Databricks UI**. This approach provides a user-friendly interface for configuring your index with managed embeddings.

**Steps to Create the Index via UI:**

1. In the left sidebar, click **Catalog** to open Catalog Explorer.
2. Navigate to your catalog and schema.
3. Find and select your Delta table (`docs_chunked_lab_3`).
4. Click **Create** (top right) and choose **Vector search index**.
5. In the dialog, configure the following settings:
   * **Name:** Enter name of the index as `docs_chunked_lab_index`
   * **Primary key:** Select `id` (the unique identifier column)
   * **Columns to sync:** Leave blank to sync all columns
   * **Embedding source:** Choose **Compute embeddings**
     - **Embedding source column:** Select `chunk`
     - **Embedding model:** Choose `databricks-gte-large-en`
   * **Sync computed embeddings:** Toggle **OFF** to save embeddings to the table
   * **Vector search endpoint:** Select your endpoint. Note that you should have this endpoint ready.
   * **Sync mode:** Choose **Triggered** (manual sync)
6. Click **Create** and monitor index creation progress.

**⏱️ Wait Time:** Index creation typically takes 2-3 minutes. You can monitor progress in the UI.

Once your index is created, proceed to the next task.

## Task 3: Get Vector Search Index Details

Get the vector search index you just created and show the details.

In [0]:
# Get the vector search index for performing searches

from databricks.vector_search.client import VectorSearchClient

# Define the index name that you created via UI
index_name = f"{catalog}.{schema}.docs_chunked_lab_index"

# Initialize the Vector Search client for later use
vsc = VectorSearchClient(disable_notice=True)

# Get the index using the vector search client
index = vsc.<FILL_IN>

# Display index information
print(index.describe())

In [0]:
%skip
# Get the vector search index for performing searches
from databricks.vector_search.client import VectorSearchClient

# Define the index name that you created via UI
index_name = f"{catalog}.{schema}.docs_chunked_lab_index"

# Initialize the Vector Search client for later use
vsc = VectorSearchClient(disable_notice=True)

# Get the index using the vector search client
index = vsc.get_index(index_name=index_name)

# Display index information
print(index.describe())

## Task 4: Similarity Search with Reranking

In this section, you will **perform similarity search with reranking** to improve retrieval precision. Reranking applies a secondary scoring step to prioritize the most contextually relevant results.

**Steps:**
1. Perform similarity search with reranking for improved precision.
1. Ask this question: `"How does the motion controller maintain balance during rapid movement?"`
1. Return 3 results.
1. Analyze the results to understand the impact of reranking.

Complete the code below to perform this task.

In [0]:
# Perform similarity search with reranking for improved precision

from databricks.vector_search.reranker import DatabricksReranker

query_text = "How does the motion controller maintain balance during rapid movement?"

# Perform similarity search with reranking
reranked_results = index.<FILL_IN>

print("=== Similarity Search with Reranking Results ===")
display(reranked_results)

In [0]:
%skip
# Perform similarity search with reranking for improved precision

from databricks.vector_search.reranker import DatabricksReranker

query_text = "How does the motion controller maintain balance during rapid movement?"

# Perform similarity search with reranking
reranked_results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=3,
    reranker=DatabricksReranker(columns_to_rerank=["chunk"])
)

print("=== Similarity Search with Reranking Results ===")
display(reranked_results)

## Task 5: Hybrid Search with Filters – Targeted Document Search

In this section, you will **implement hybrid search with filtering** to combine semantic similarity, keyword matching, and document targeting. This approach delivers highly precise results by leveraging multiple search strategies together.

Suppose you want to answer: **"Find procedures that describe battery replacement for the A1 model."**

- *Pure semantic (similarity) search* may return passages about battery charging or thermal management, since they are semantically related.
- *Adding a keyword filter* for "Battery" or "A1" narrows the results to relevant document sections.
- *Filtering by file name* further ensures you retrieve only the procedures from the correct document.

**Steps:**
1. Perform a **hybrid search** for the query.
1. **Filter results** to only those from `05_Orion_Maintenance_and_Servicing_Guide_v3.pdf`.
1. Return **2 records** with **all columns**.
1. Analyze how combining hybrid search and filters improves search precision.

Complete the code below to perform this task.

In [0]:
# Perform hybrid search with filtering to target specific documents

query_text = "Find procedures that describe battery replacement for the A1 model."

# Perform hybrid search with document path filter
filtered_hybrid_results = index.<FILL_IN>

print("=== Hybrid Search with Filters Results ===")
display(filtered_hybrid_results)

In [0]:
%skip
# Perform hybrid search with filtering to target specific documents

query_text = "Find procedures that describe battery replacement for the A1 model."

# Perform hybrid search with document path filter
filtered_hybrid_results = index.similarity_search(
    query_text=query_text,
    columns=["id","path", "chunk"],
    query_type="hybrid",
    filters={"path LIKE": "05_Orion_Maintenance_and_Servicing_Guide_v3.pdf"},  # Filter by path containing safety-related documents
    num_results=5
)

print("=== Hybrid Search with Filters Results ===")
display(filtered_hybrid_results)

**💡 Analysis & Reflection:**

**Which search method would you choose for:**
- A technical documentation system where users search for specific procedures?
- A legal repository where precision is critical?
- A customer support knowledge base with varied query types?

**Think About:** Based on your results, which approach would you recommend for a production RAG system and why?

## Summary and Next Steps

You have completed the lab to build a vector search solution for document retrieval using Databricks. You learned how to:

* **Prepare** document data by reading from parquet and creating a Delta table with Change Data Feed enabled.
* **Create** a vector search index using the Databricks UI with managed embeddings.
* **Implement** similarity search with reranking to improve retrieval precision.
* **Execute** hybrid search combining semantic similarity with keyword matching.
* **Apply** filters to target specific documents and narrow search scope.

**Next Steps (Optional):**
* Explore different embedding models and experiment with **multilingual and domain-specific** embedding models.
* Investigate how endpoint configuration, embedding dimension, and refresh mode affect latency and cost.
* Investigate how to retrieve rows based on user's credentials. 

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>